# M1 · Recompute the frozen baseline

**Question:** Is 55.21 s a repeatable warm latency, or a cherry-picked sample? Read the two pinned B0 sessions; recompute the median without borrowing any smoke/cold/fresh record.

**Learning loop:** predict → inspect frozen evidence → compute → critique → propose a controlled experiment.

Reference lesson: [open course/M01_baseline/LESSON.md](../course/M01_baseline/LESSON.md). Switch language: [中文](./01_baseline_and_uncertainty.zh-CN.ipynb).

**Default mode:** CPU-only, internet-free, read-only. GPU work is never run implicitly.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root (see notebooks/README).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab import notebook_support as ns
import statistics
print("Course repository:", ROOT)
print("Mode: OFFLINE / committed reference evidence; NO GPU work is launched by default.")


## Inspect and recompute real evidence

Run cells in order. Explain every denominator and scope.

In [ ]:
raw = ns.reference_warm("B0")
claimed = ns.summary_rows()["B0"]
observed = statistics.median(raw)
assert len(raw) == claimed["n_measure"] == 18
assert abs(observed - claimed["warm_total_median_s"]) < 0.03
print(f"B0 reference: n={len(raw)}, median={observed:.3f}s, min={min(raw):.3f}s, max={max(raw):.3f}s")
print("Do NOT label these samples as a new physical GPU run.")
plot_values = {"B0 p50": observed, "min": min(raw), "max": max(raw)}
plot_ylabel, plot_title = "seconds", "Measured B0 warm records"


In [ ]:
SHOW_PLOT = False  # Optional: requires matplotlib in the notebook kernel
if SHOW_PLOT:
    import matplotlib.pyplot as plt
    labels = list(plot_values)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(labels, [plot_values[label] for label in labels])
    ax.set_ylabel(plot_ylabel)
    ax.set_title(plot_title)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Optional plot skipped; set SHOW_PLOT=True to visualize the measured values.")


## Optional physical GPU smoke — explicit opt-in

Requires the staged ROCm environment and sufficient model/GPU resources. Never used in CPU CI. The run goes to `runs/`, not official evidence.

In [ ]:
RUN_GPU = False  # Only switch on after checking platform quota + available models
if RUN_GPU:
    # Also set QIR_NOTEBOOK_GPU_ACK=YES in your own kernel environment.
    # The helper enforces --smoke and writes under runs/, never evidence/.
    ns.gpu_smoke("B0", "notebook-b0", confirmed=True)
else:
    print("GPU NOT_RUN; evidence analysis above is valid without GPU access.")


## Your turn / Self-check

Recompute the B0 median using only one of the two sessions. Why does reporting both sessions improve transparency? Explain `n=18` vs `n=9` for other arms.

**Evidence discipline:** Answer in your own words. This notebook's numbers come from existing committed records, not your GPU. If your optional smoke runs, its data must remain under `runs/` and may not be presented as a full-protocol median.
